[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_16/MFM_ch16_pagnotta/MFM_ch16_pagnotta.ipynb)

# MFM_ch16_pagnotta

Case study, Pagnotta (2022, Review of Financial Studies), Decentralizing money: Bitcoin prices and blockchain security: Lemma 1 (hashrate proportional to the expected USD block reward) tested on the course data. Bitcoin closing price (course data), hashrate and daily issuance (Coin Metrics Community Data), August 2010 - September 2026: monthly log hashrate on the log USD block reward and a linear trend (Newey-West errors, Engle-Granger test), in the paper's window (August 2010 - January 2020) and out of sample (February 2020 - August 2026); hashrate around the four halvings (2012, 2016, 2020, 2024), against the -50% predicted at a constant price.

*Modelling Financial Markets (MFM), Chapter 16: Digital Assets and DeFi. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_16'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import json
import time
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
import warnings
warnings.filterwarnings('ignore')


## Style

In [ ]:
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none',
                     'savefig.transparent': True, 'axes.grid': False, 'font.family': 'sans-serif',
                     'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'], 'font.size': 9,
                     'axes.labelsize': 10, 'axes.titlesize': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.6, 'lines.linewidth': 1.1,
                     'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.fontsize': 8})
MainBlue, IDAred, Forest, Orange, Purple, Teal, Navy = ('#1A3A6E', '#CD0000', '#2E7D32', '#E67E22', '#8E44AD',
                                                       '#17A2B8', '#1F2A44')


## Data

Bitcoin closing price from the course data (`data/market/BTC-USD.CC.csv`, read from GitHub when the folder is not found); hashrate (`HashRate`, TH/s) and new coins issued per day (`IssTotNtv`) from Coin Metrics Community Data (no key).

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'
_CACHE = {}
UA = {'User-Agent': 'Mozilla/5.0 (MFM course notebook)'}


def _get_json(url, tries=4):
    """Read a JSON response."""
    for i in range(tries):
        try:
            return json.load(urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=120))
        except Exception:
            if i == tries - 1:
                raise
            time.sleep(3 * (i + 1))


def price(key='BTC', start=None, end=END):
    """Daily closing price of Bitcoin (7 days a week) from the course data."""
    path = os.path.join(MARKET_DIR, 'BTC-USD.CC.csv')
    src = path if os.path.exists(path) else REPO_RAW + 'BTC-USD.CC.csv'
    s = pd.read_csv(src, index_col='date', parse_dates=True).sort_index()['close'].loc[start:end]
    return s[s > 0].dropna().rename(key)


def coin_metrics(asset, metrics, start='2010-01-01', end=END):
    """Daily network metrics of a crypto-asset from Coin Metrics Community Data, e.g. HashRate (TH/s, estimated from
    block times and difficulty) and IssTotNtv (new coins issued that day = block rewards, in native units)."""
    key = ('cm', asset, tuple(metrics), start, end)
    if key in _CACHE:
        return _CACHE[key]
    url = ('https://community-api.coinmetrics.io/v4/timeseries/asset-metrics?assets=' + asset + '&metrics=' +
           ','.join(metrics) + f'&frequency=1d&start_time={start}&end_time={end}&page_size=10000')
    rows = []
    while url:
        js = _get_json(url)
        rows += js.get('data', [])
        url = js.get('next_page_url')
    df = pd.DataFrame({m: {pd.Timestamp(r['time'][:10]): float(r[m]) for r in rows if r.get(m) is not None}
                       for m in metrics}).sort_index()
    _CACHE[key] = df
    return df


## Analysis

In [ ]:
OUT = OUT_DIR
HALVINGS = ['2012-11-28', '2016-07-09', '2020-05-11', '2024-04-20']
HCOL = {'2012-11-28': Teal, '2016-07-09': Forest, '2020-05-11': Purple, '2024-04-20': IDAred}
PAPER = ('2010-08', '2020-01')          # Figure 1 of the paper: August 2010 - January 2020
OOS = ('2020-02', '2026-08')            # complete months after the paper's window, up to END
K_MIN, K_MAX, LAGS = -30, 90, 12


def daily():
    """Daily price (course data), hashrate (TH/s), issuance (BTC) and USD block reward, on common days."""
    cm = coin_metrics('btc', ['HashRate', 'IssTotNtv'], '2010-07-01', END)
    d = pd.concat([price('BTC').rename('P'), cm['HashRate'].rename('H'), cm['IssTotNtv'].rename('R')],
                  axis=1, join='inner').dropna()
    d = d[(d.H > 0) & (d.R > 0)]
    d['B'] = d.P * d.R
    return d


def levels(d):
    """log H on log B and a trend, monthly means; HAC errors; Engle-Granger p-value."""
    m = np.log(d[['P', 'H', 'B']].resample('ME').mean())
    out = {}
    for name, (a, b) in [('paper', PAPER), ('oos', OOS)]:
        z = m.loc[a:b]
        t = np.arange(len(z)) / 12
        X = sm.add_constant(pd.DataFrame({'B': z.B, 't': t}, index=z.index))
        r = sm.OLS(z.H, X).fit(cov_type='HAC', cov_kwds={'maxlags': LAGS})
        r0 = sm.OLS(z.H, sm.add_constant(z.B)).fit()
        out[name] = dict(first=str(z.index[0].date())[:7], last=str(z.index[-1].date())[:7], n=len(z),
                         b=r.params['B'], se=r.bse['B'], trend=100 * r.params['t'], r2=r.rsquared,
                         b_notrend=r0.params['B'], r2_notrend=r0.rsquared,
                         eg_p=coint(z.H, z.B, trend='ct')[1])
    return out, m


def event(d):
    """Change (%) of the 7-day mean hashrate and USD block reward from the 7 days before each halving."""
    H7, B7 = d.H.rolling(7).mean(), d.B.rolling(7).mean()
    ks = np.arange(K_MIN, K_MAX + 1)
    paths, out = {}, {}
    for h in HALVINGS:
        t0 = pd.Timestamp(h)
        base = t0 - pd.Timedelta(days=1)
        idx = [t0 + pd.Timedelta(days=int(k)) for k in ks]
        ph = 100 * (H7.reindex(idx).values / H7[base] - 1)
        pb = 100 * (B7.reindex(idx).values / B7[base] - 1)
        paths[h] = dict(k=ks, H=ph, B=pb)
        out[h] = {f'H{k}': ph[k - K_MIN] for k in (30, 90)}
        out[h].update({f'B{k}': pb[k - K_MIN] for k in (30, 90)})
        post = ks >= 0
        j = int(np.nanargmin(np.where(post, ph, np.nan)))
        out[h]['H_min'], out[h]['H_min_day'] = ph[j], int(ks[j])
        out[h]['R_cut'] = 100 * (d.R.loc[t0 + pd.Timedelta(days=1):t0 + pd.Timedelta(days=7)].mean() /
                                 d.R.loc[t0 - pd.Timedelta(days=7):base].mean() - 1)
    return out, paths


In [ ]:
def fig(m, paths):
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), gridspec_kw={'width_ratios': [1.25, 1]})
    ax = axes[0]
    mm = np.exp(m)
    ax.plot(mm.index, mm.P, color=Orange, lw=1.1, label='Bitcoin price (USD, left axis, monthly mean)')
    ax.set_yscale('log')
    ax.set_ylabel('Price (USD)')
    ax2 = ax.twinx()
    ax2.spines['right'].set_visible(True)
    ax2.plot(mm.index, mm.H / 1e6, color=MainBlue, lw=1.1, label='Hashrate (EH/s, right axis, monthly mean)')
    ax2.set_yscale('log')
    for h in HALVINGS:
        ax.axvline(pd.Timestamp(h), color=Navy, lw=0.6, ls=':')
    ax.axvline(pd.Timestamp('2020-01-31'), color=Navy, lw=0.8, ls='--')
    ax.text(pd.Timestamp('2020-01-31'), 0.04, ' end of the\n paper\'s Figure 1', transform=ax.get_xaxis_transform(),
            fontsize=7, color='black', va='bottom')
    ax.set_title('Price and hashrate, log scales; dotted: halvings', fontsize=9)
    ax = axes[1]
    for h in HALVINGS:
        p = paths[h]
        ax.plot(p['k'], p['H'], color=HCOL[h], lw=1.1, label=f'Hashrate, halving of {h[:4]}')
    ax.axhline(-50, color=Navy, lw=0.8, ls='--', label='Lemma 1 at a constant price: -50%')
    ax.axhline(0, color=Navy, lw=0.4)
    ax.axvline(0, color=Navy, lw=0.6, ls=':')
    ax.set_xlabel('Days from the halving')
    ax.set_ylabel('Change (%)')
    ax.set_title('Hashrate around the halvings, change\nfrom the week before (7-day means)', fontsize=9)
    ax.set_xlim(K_MIN, K_MAX)
    fig.tight_layout(w_pad=2.5)
    h1, l1 = axes[0].get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    h3, l3 = axes[1].get_legend_handles_labels()
    fig.legend(h1 + h2 + h3, l1 + l2 + l3, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=3, frameon=False)
    for ext, kw in (('pdf', {}), ('png', {'dpi': 180})):
        plt.savefig(os.path.join(OUT, f'ch16_pagnotta_hashrate.{ext}'), bbox_inches='tight', transparent=True, **kw)
    plt.show()
    print('   saved ch16_pagnotta_hashrate')


def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, np.integer):
        return int(x)
    return x


## Run

In [ ]:
d = daily()
lv, m = levels(d)
ev, paths = event(d)
fig(m.loc[PAPER[0]:], paths)
res = dict(levels=lv, halvings=ev, h_last=d.H.iloc[-7:].mean() / 1e6)
with open(os.path.join(OUT, 'ch16_pagnotta.json'), 'w') as fh:
    json.dump(jsonable(res), fh, indent=1)
print(json.dumps(jsonable(res), indent=1))


![ch16_pagnotta_hashrate](./ch16_pagnotta_hashrate.png)

Output: `ch16_pagnotta_hashrate.pdf`, `ch16_pagnotta.json`